<a href="https://colab.research.google.com/github/beoko/calculadora/blob/main/Fundos_CVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install requests beautifulsoup4

In [21]:
!pip -q install openpyxl

In [14]:
!pip -q install pandas requests

In [49]:
!pip -q install pandas requests openpyxl

import re, io, zipfile, requests, pandas as pd
import csv as pycsv
from typing import Set, List, Tuple

pycsv.field_size_limit(10_000_000)

CDA_DIR_URL = "https://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def norm_cols(cols):
    return [
        str(c).upper().strip()
          .replace("\ufeff", "")
          .replace("\r", "")
          .replace("\n", "")
        for c in cols
    ]

def norm_isin(x: str) -> str:
    return re.sub(r"[^A-Z0-9]", "", str(x).upper())

def norm_cnpj(x: str) -> str:
    return re.sub(r"\D", "", str(x))

def descobrir_zip_mais_recente():
    html = requests.get(CDA_DIR_URL, headers=HEADERS, timeout=60).text
    zips = re.findall(r'cda_fi_(\d{6})\.zip', html)
    if not zips:
        raise RuntimeError("Não encontrei cda_fi_YYYYMM.zip no diretório CDA.")
    yyyymm = max(zips)
    return yyyymm, f"{CDA_DIR_URL}cda_fi_{yyyymm}.zip"

def iter_chunks_pandas(zf: zipfile.ZipFile, name: str, chunksize=200_000):
    raw = zf.open(name, "r")
    text = io.TextIOWrapper(raw, encoding="latin1", errors="replace", newline="")
    # engine python + on_bad_lines skip ajuda muito, mas alguns arquivos ainda quebram
    return pd.read_csv(
        text,
        sep=";",
        chunksize=chunksize,
        engine="python",
        on_bad_lines="skip",
        quotechar='"',
        escapechar='\\'
    )

def scan_with_pandas(zf: zipfile.ZipFile, name: str, isin: str) -> Tuple[Set[str], bool]:
    """
    Retorna: (cnpjs, found_any_match)
    """
    found = False
    cnpjs: Set[str] = set()

    for chunk in iter_chunks_pandas(zf, name):
        chunk.columns = norm_cols(chunk.columns)

        isin_cols = [c for c in chunk.columns if "ISIN" in c]
        if not isin_cols:
            continue

        mask = False
        for c in isin_cols:
            mask = mask | (chunk[c].astype(str).map(norm_isin) == isin)
        sub = chunk.loc[mask]
        if sub.empty:
            continue

        found = True

        # pega coluna de CNPJ preferida
        cnpj_col = None
        if "CNPJ_FUNDO_CLASSE" in sub.columns:
            cnpj_col = "CNPJ_FUNDO_CLASSE"
        else:
            cnpj_candidates = [c for c in sub.columns if "CNPJ" in c]
            if cnpj_candidates:
                # prioriza alguma que contenha CLASSE se existir
                classe_first = [c for c in cnpj_candidates if "CLASSE" in c]
                cnpj_col = (classe_first[0] if classe_first else cnpj_candidates[0])

        if cnpj_col:
            vals = sub[cnpj_col].astype(str).map(norm_cnpj).tolist()
            cnpjs.update(v for v in vals if v and v != "NAN")

    return cnpjs, found

def scan_with_csv_fallback(zf: zipfile.ZipFile, name: str, isin: str) -> Tuple[Set[str], bool]:
    """
    Fallback MUITO tolerante: lê linha a linha com csv.reader.
    Procura:
      - alguma coluna com 'ISIN' no cabeçalho
      - alguma coluna com 'CNPJ' no cabeçalho (prioriza CNPJ_FUNDO_CLASSE)
    """
    cnpjs: Set[str] = set()
    found = False

    raw = zf.open(name, "r")
    text = io.TextIOWrapper(raw, encoding="latin1", errors="replace", newline="")
    reader = pycsv.reader(text, delimiter=';')

    try:
        header = next(reader)
    except StopIteration:
        return cnpjs, found

    header_n = norm_cols(header)

    isin_idx = [i for i, h in enumerate(header_n) if "ISIN" in h]
    cnpj_candidates = [i for i, h in enumerate(header_n) if "CNPJ" in h]

    # se não tem coluna ISIN, não tem como bater
    if not isin_idx:
        return cnpjs, found

    # escolhe melhor coluna de CNPJ
    cnpj_best = None
    if "CNPJ_FUNDO_CLASSE" in header_n:
        cnpj_best = header_n.index("CNPJ_FUNDO_CLASSE")
    elif cnpj_candidates:
        # prioriza alguma com CLASSE
        classe_idxs = [i for i in cnpj_candidates if "CLASSE" in header_n[i]]
        cnpj_best = classe_idxs[0] if classe_idxs else cnpj_candidates[0]

    # varre linhas
    for row in reader:
        if not row:
            continue

        # garante tamanho mínimo (csv quebrado às vezes vem menor)
        # vamos proteger acesso por índice
        def safe_get(i):
            return row[i] if i < len(row) else ""

        # bate ISIN em qualquer coluna ISIN
        matched = False
        for i in isin_idx:
            if norm_isin(safe_get(i)) == isin:
                matched = True
                break

        if not matched:
            continue

        found = True
        if cnpj_best is not None:
            c = norm_cnpj(safe_get(cnpj_best))
            if c:
                cnpjs.add(c)

    return cnpjs, found

def main():
    print("🔎 Busca CDA por ISIN (robusta)")
    isin = norm_isin(input("Informe o ISIN (ex: BRBRKMDBS0A1): ").strip())

    yyyymm, zip_url = descobrir_zip_mais_recente()
    print(f"\n📦 Baixando CDA mais recente: {zip_url}")

    r = requests.get(zip_url, headers=HEADERS, timeout=180)
    r.raise_for_status()

    zf = zipfile.ZipFile(io.BytesIO(r.content))
    csvs = [n for n in zf.namelist() if n.lower().endswith(".csv")]

    print(f"🗂️ CSVs no ZIP: {len(csvs)}")
    print(f"🔎 Procurando ISIN = {isin}")
    print("✅ Vou varrer TODOS os CSVs; se pandas falhar, uso fallback csv.reader.\n")

    all_cnpjs: Set[str] = set()
    arquivos_com_match: List[str] = []
    arquivos_com_erro: List[Tuple[str,str]] = []

    for name in csvs:
        print(f"➡️ Lendo: {name}")
        try:
            cnpjs, found = scan_with_pandas(zf, name, isin)
        except Exception as e:
            # fallback tolerante
            try:
                cnpjs, found = scan_with_csv_fallback(zf, name, isin)
            except Exception as e2:
                arquivos_com_erro.append((name, f"{e} | fallback: {e2}"))
                print(f"⚠️ Erro (pandas e fallback): {e2}")
                continue

        if found:
            arquivos_com_match.append(name)
        all_cnpjs.update(cnpjs)
        print("— ok")

    cnpjs_list = sorted(c for c in all_cnpjs if c and c != "NAN")

    print("\n================ RESUMO ================\n")
    print(f"Arquivos lidos: {len(csvs)}")
    print(f"Arquivos com match: {len(arquivos_com_match)}")
    for a in arquivos_com_match:
        print(" -", a)

    if arquivos_com_erro:
        print(f"\nArquivos com erro total: {len(arquivos_com_erro)} (não impacta os demais)")
        for n, err in arquivos_com_erro[:5]:
            print(" -", n, "=>", err)

    if not cnpjs_list:
        print("\n❌ Nenhum CNPJ encontrado para esse ISIN neste mês.")
        print("👉 Pode ser que esse ISIN não esteja presente no CDA_YYYYMM atual.")
        return

    print("\n================ CNPJs ENCONTRADOS ================\n")
    print(f"Total: {len(cnpjs_list)}\n")
    print("\n".join(cnpjs_list))

    # Export XLSX
    out = pd.DataFrame({"CNPJ_ENCONTRADO": cnpjs_list})
    xlsx_name = f"cnpjs_por_isin_{isin}_{yyyymm}.xlsx"
    with pd.ExcelWriter(xlsx_name, engine="openpyxl") as writer:
        out.to_excel(writer, index=False, sheet_name="cnpjs")
        pd.DataFrame({"ARQUIVOS_COM_MATCH": arquivos_com_match}).to_excel(writer, index=False, sheet_name="arquivos_com_match")
        if arquivos_com_erro:
            pd.DataFrame(arquivos_com_erro, columns=["ARQUIVO", "ERRO"]).to_excel(writer, index=False, sheet_name="arquivos_com_erro")

    print(f"\n📊 XLSX salvo: {xlsx_name}")

    try:
        from google.colab import files
        files.download(xlsx_name)
    except Exception:
        pass

main()


🔎 Busca CDA por ISIN (robusta)
Informe o ISIN (ex: BRBRKMDBS0A1): BRVLMEDBS007

📦 Baixando CDA mais recente: https://dados.cvm.gov.br/dados/FI/DOC/CDA/DADOS/cda_fi_202512.zip
🗂️ CSVs no ZIP: 12
🔎 Procurando ISIN = BRVLMEDBS007
✅ Vou varrer TODOS os CSVs; se pandas falhar, uso fallback csv.reader.

➡️ Lendo: cda_fie_202512.csv
— ok
➡️ Lendo: cda_fie_CONFID_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_1_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_2_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_3_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_4_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_5_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_6_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_7_202512.csv
— ok
➡️ Lendo: cda_fi_BLC_8_202512.csv
— ok
➡️ Lendo: cda_fi_CONFID_202512.csv
— ok
➡️ Lendo: cda_fi_PL_202512.csv
— ok

================ RESUMO ================

Arquivos lidos: 12
Arquivos com match: 1
 - cda_fi_BLC_4_202512.csv

================ CNPJs ENCONTRADOS ================

Total: 51

00888897000131
01214092000175
10326625000100
13106998000155
20

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>